# Takeda Pharmaceuticals — Data Science Take-Home Challenge

## Compensation Equity & Workforce Analytics

**Role:** Senior/Staff Data Scientist — Global Compensation & Benefits  
**Delivered by:** IBM Consulting  
**Time Limit:** 72 hours  
**Difficulty:** PhD-level

---

## Context

You are a data scientist on the Global Compensation & Benefits team at Takeda Pharmaceuticals. The Chief People Officer has raised concerns about potential gender-based pay inequities across the company's global workforce. Your task is to conduct a rigorous, end-to-end analysis — from data quality assessment to actionable recommendations.

You will work with a synthetic dataset of ~5,000 employees across 10 countries. The data may contain noise, outliers, missing values, and structural biases that mirror real-world HR data challenges.

## What We're Evaluating

| Dimension | Weight | What We Look For |
|-----------|--------|-----------------|
| **Statistical Rigor** | 30% | Correct methodology, assumptions testing, effect sizes, confidence intervals |
| **Data Engineering** | 20% | Preprocessing, feature engineering, handling of messy data |
| **EDA & Visualization** | 20% | Insightful exploratory analysis, publication-quality plots |
| **Modeling** | 20% | Model selection, validation, interpretation |
| **Communication** | 10% | Clear narrative, executive-ready insights |

## Rules
- Use Python (pandas, numpy, scikit-learn, statsmodels, matplotlib/seaborn/plotly)
- Show all work — we value the process as much as the result
- Comment your code explaining **why**, not just what
- Treat this as a deliverable to a non-technical executive team

In [ ]:
!pip3 install -q pandas numpy scikit-learn statsmodels matplotlib seaborn scipy plotly

## Part 0: Data Generation

Run the cell below to generate your dataset. **Do not modify this cell.** The dataset includes intentional imperfections that you must handle.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def generate_challenge_data(seed=2024):
    """Generate messy, realistic HR compensation data for the take-home challenge."""
    rng = np.random.default_rng(seed)
    N = 5200
    
    COUNTRIES = {
        "US": {"mult": 1.0, "gap": 0.08, "n_frac": 0.25},
        "JP": {"mult": 0.75, "gap": 0.12, "n_frac": 0.14},
        "DE": {"mult": 0.90, "gap": 0.06, "n_frac": 0.10},
        "UK": {"mult": 0.85, "gap": 0.09, "n_frac": 0.08},
        "CH": {"mult": 1.15, "gap": 0.05, "n_frac": 0.05},
        "BR": {"mult": 0.45, "gap": 0.10, "n_frac": 0.08},
        "CN": {"mult": 0.50, "gap": 0.11, "n_frac": 0.10},
        "IN": {"mult": 0.30, "gap": 0.14, "n_frac": 0.08},
        "SG": {"mult": 0.80, "gap": 0.07, "n_frac": 0.06},
        "AU": {"mult": 0.88, "gap": 0.06, "n_frac": 0.06},
    }
    
    BASE_BY_LEVEL = {1: 45000, 2: 55000, 3: 68000, 4: 82000, 5: 100000,
                     6: 125000, 7: 155000, 8: 195000, 9: 250000, 10: 330000}
    
    DEPTS = {"R&D": (0.30, 1.10), "Commercial": (0.25, 1.05), 
             "Manufacturing": (0.20, 0.95), "Corporate": (0.15, 1.00), 
             "Medical Affairs": (0.10, 1.08)}
    
    EDU_PREMIUM = {"Bachelor's": 0.0, "Master's": 0.08, "PhD": 0.18, "MD": 0.25, "MBA": 0.15}
    
    countries = list(COUNTRIES.keys())
    country_probs = [COUNTRIES[c]["n_frac"] for c in countries]
    depts = list(DEPTS.keys())
    dept_probs = [DEPTS[d][0] for d in depts]
    edus = list(EDU_PREMIUM.keys())
    edu_probs = [0.25, 0.30, 0.20, 0.10, 0.15]
    
    records = []
    for i in range(N):
        eid = f"EMP-{i+1:05d}"
        gender = rng.choice(["M", "F"], p=[0.55, 0.45])
        country = rng.choice(countries, p=country_probs)
        dept = rng.choice(depts, p=dept_probs)
        edu = rng.choice(edus, p=edu_probs)
        
        # Level with gender bias
        if gender == "F":
            lp = np.array([0.12, 0.14, 0.16, 0.15, 0.14, 0.12, 0.08, 0.05, 0.03, 0.01])
        else:
            lp = np.array([0.08, 0.10, 0.13, 0.14, 0.15, 0.14, 0.11, 0.08, 0.05, 0.02])
        level = rng.choice(range(1, 11), p=lp)
        
        yrs = max(0, min(40, int(rng.normal(level * 2.5, 3))))
        
        # Performance rating (1-5, slight gender bias)
        if gender == "F":
            perf = rng.choice([1,2,3,4,5], p=[0.02, 0.08, 0.30, 0.40, 0.20])
        else:
            perf = rng.choice([1,2,3,4,5], p=[0.02, 0.06, 0.25, 0.40, 0.27])
        
        # Tenure at company
        tenure = max(0, min(yrs, int(rng.normal(yrs * 0.6, 2))))
        
        # Hire date
        hire_year = 2024 - tenure
        hire_month = rng.integers(1, 13)
        hire_date = f"{hire_year}-{hire_month:02d}-01"
        
        # Salary calculation
        base = BASE_BY_LEVEL[level]
        base *= COUNTRIES[country]["mult"]
        base *= DEPTS[dept][1]
        base *= (1 + EDU_PREMIUM[edu])
        base *= (1 + yrs * 0.012)
        base *= (1 + (perf - 3) * 0.03)  # performance adjustment
        
        if gender == "F":
            gap = max(0, rng.normal(COUNTRIES[country]["gap"], COUNTRIES[country]["gap"] * 0.3))
            base *= (1 - gap)
        
        base *= rng.normal(1.0, 0.05)
        base_salary = round(max(base, 20000), 2)
        
        bonus_pct = 0.05 + (level - 1) * 0.028
        bonus = round(base_salary * bonus_pct * rng.uniform(0.6, 1.4), 2)
        
        records.append({
            "employee_id": eid, "gender": gender, "country": country,
            "department": dept, "job_level": level, "education": edu,
            "years_experience": yrs, "tenure_years": tenure,
            "performance_rating": perf, "hire_date": hire_date,
            "base_salary": base_salary, "bonus": bonus,
            "total_compensation": round(base_salary + bonus, 2),
        })
    
    df = pd.DataFrame(records)
    
    # === INJECT DATA QUALITY ISSUES ===
    
    # 1. Missing values (~3% scattered)
    for col in ["education", "performance_rating", "years_experience", "bonus"]:
        mask = rng.random(N) < 0.03
        df.loc[mask, col] = np.nan
    
    # 2. Duplicate rows (~1%)
    dup_idx = rng.choice(N, size=50, replace=False)
    dups = df.iloc[dup_idx].copy()
    df = pd.concat([df, dups], ignore_index=True)
    
    # 3. Outliers: a few extreme salaries (data entry errors)
    outlier_idx = rng.choice(len(df), size=15, replace=False)
    df.loc[outlier_idx[:8], "base_salary"] *= rng.uniform(3, 8, size=8)
    df.loc[outlier_idx[8:], "base_salary"] *= rng.uniform(0.05, 0.2, size=7)
    df.loc[outlier_idx, "total_compensation"] = df.loc[outlier_idx, "base_salary"] + df.loc[outlier_idx, "bonus"].fillna(0)
    
    # 4. Inconsistent encoding
    gender_map = {"M": ["M", "Male", "male", "m"], "F": ["F", "Female", "female", "f"]}
    noise_idx = rng.choice(len(df), size=80, replace=False)
    for idx in noise_idx:
        g = df.loc[idx, "gender"]
        if g in gender_map:
            df.loc[idx, "gender"] = rng.choice(gender_map[g])
    
    # 5. A few negative bonus values (refunds/clawbacks — realistic but need handling)
    neg_idx = rng.choice(len(df), size=10, replace=False)
    df.loc[neg_idx, "bonus"] = -abs(df.loc[neg_idx, "bonus"])
    
    # 6. Shuffle and reset index
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return df

# Generate the dataset
df = generate_challenge_data()
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

---

## Part 1: Data Quality Assessment & Preprocessing (20 points)

**Tasks:**

1. **Profile the dataset** — Summarize dimensions, data types, missing values, and basic statistics. Identify all data quality issues present.

2. **Clean the data** — Handle each issue you identified (duplicates, missing values, inconsistent encodings, outliers, etc.). For each decision, explain your reasoning and trade-offs.

3. **Feature engineering** — Create at least 3 new features that could be useful for compensation analysis. Examples might include: compensation ratios, experience-adjusted metrics, country-normalized salaries, etc.

4. **Validation** — After cleaning, show that your processed dataset is internally consistent (e.g., total_compensation = base_salary + bonus, no impossible values).

**Expectations:**
- We expect you to find **at least 5 distinct data quality issues**
- Justify whether you impute, drop, or flag each issue
- Show before/after comparisons

In [ ]:
# YOUR CODE HERE — Part 1


---

## Part 2: Exploratory Data Analysis (20 points)

**Tasks:**

1. **Univariate analysis** — Distribution of compensation, job levels, tenure, and performance across the workforce. Use appropriate visualizations.

2. **Bivariate analysis** — Explore relationships between compensation and each predictor variable. Pay special attention to gender differences at each level of analysis.

3. **Multivariate exploration** — Create at least 2 multi-dimensional visualizations that reveal patterns not visible in simpler plots. Consider heatmaps, pair plots, faceted plots, or interactive visualizations.

4. **Key findings** — Write a bulleted summary of your top 5 EDA findings. For each, state the finding, the evidence, and why it matters for the pay gap analysis.

**Expectations:**
- At least 8 distinct, well-labeled visualizations
- At least one visualization must be publication-quality (suitable for a board presentation)
- Statistical summaries should accompany visual findings (medians, IQR, effect sizes)

In [ ]:
# YOUR CODE HERE — Part 2


---

## Part 3: Statistical Modeling — Pay Gap Decomposition (30 points)

This is the core of the challenge. You must build a rigorous model to quantify the gender pay gap.

**Tasks:**

### 3a. Baseline Model (10 points)
- Fit a log-linear regression: $\ln(\text{TC}) = X\beta + \alpha \cdot \mathbf{1}_{\text{Female}} + \epsilon$
- Control for: job_level, years_experience, education, country, department, performance_rating, tenure
- Report and interpret the gender coefficient $\alpha$ with 95% confidence interval
- Test regression assumptions (normality of residuals, heteroscedasticity, multicollinearity via VIF)

### 3b. Advanced Modeling (10 points)
Choose **at least one** advanced approach and compare to baseline:

| Approach | What It Tests |
|----------|--------------|
| **Quantile Regression** (τ = 0.10, 0.25, 0.50, 0.75, 0.90) | Is the gap larger at the top (glass ceiling) or bottom (sticky floor)? |
| **Hierarchical/Mixed-Effects Model** | Country-level random intercepts and slopes for gender |
| **Blinder-Oaxaca Decomposition** | Formally decompose into endowment, coefficient, and interaction effects |
| **Propensity Score Matching** | Match male-female pairs on observables, estimate ATT |

### 3c. Interaction Effects (5 points)
- Test at least 2 interaction terms (e.g., gender × country, gender × job_level)
- Interpret the interactions: where is the gap most/least severe?
- Visualize the interaction effects

### 3d. Model Diagnostics (5 points)
- Residual analysis with plots
- Influential observations (Cook's distance, leverage)
- Cross-validation (k-fold) to assess generalization
- Compare models using AIC/BIC and adjusted R²

**Expectations:**
- Use `statsmodels` for inference (p-values, CIs) — not just scikit-learn
- Report effect sizes, not just significance
- Discuss practical vs. statistical significance

In [ ]:
# YOUR CODE HERE — Part 3


---

## Part 4: Correction Recommendations & Business Impact (20 points)

**Tasks:**

### 4a. Correction Algorithm (10 points)
- For each underpaid female employee, estimate her "fair" salary (what she would earn absent the gender effect)
- Generate a ranked correction list: employee_id, current_comp, fair_comp, recommended_raise, raise_%
- Build a budget simulator: given a budget constraint $B$, which employees get corrections first? Justify your prioritization scheme.

### 4b. Scenario Analysis (5 points)
Create a table or visualization showing:

| Budget (% of full correction) | Total Cost | Employees Corrected | Remaining Gap (%) |
|-------------------------------|------------|--------------------|--------------------|
| 25% | ? | ? | ? |
| 50% | ? | ? | ? |
| 75% | ? | ? | ? |
| 100% | ? | ? | ? |

### 4c. Executive Summary (5 points)
Write a **1-page executive summary** (in markdown) as if presenting to the Chief People Officer. Include:
- The headline finding (1 sentence)
- Key metrics (3-4 bullet points)
- Recommended actions (prioritized)
- Risks and caveats
- Estimated annual cost

**Expectations:**
- The correction algorithm should be defensible — explain why you chose your approach
- The executive summary should be understandable by a non-technical audience
- Include at least one compelling visualization for the board presentation

In [ ]:
# YOUR CODE HERE — Part 4


---

## Part 5: Bonus — Structural Equity Analysis (10 bonus points)

*Optional. Attempt only if you have time remaining.*

The pay gap has two components:
1. **Direct gap** — Women paid less for the same role
2. **Structural gap** — Women underrepresented in higher-paying roles

**Tasks:**
- Quantify the structural component: what percentage of the raw gap is explained by women being concentrated in lower job levels?
- Build a promotion velocity model: estimate the expected time-to-promotion by gender at each level. Is there a "broken rung" or "glass ceiling" effect?
- If Takeda achieved gender parity in job level distribution (while keeping current within-level pay), how much of the raw gap would close?

**Expectations:**
- This is intentionally open-ended — we want to see how you think about a novel problem
- Simulation-based approaches are welcome

In [ ]:
# YOUR CODE HERE — Part 5 (Bonus)
